# 60/40 benchmark — SPXT / LUATTRUU

Daily-rebalanced 60% S&P 500 Total Return (`SPXT Index`) / 40% Bloomberg U.S. Treasury Total Return (`LUATTRUU Index`). Raw levels are read from `data/stock_bond_pricing_data_uncleaned.xlsx`.

The backtest is aligned to the same calendar as [trend_model_method1_backtest_updated.ipynb](trend_model_method1_backtest_updated.ipynb) (intersection with `method1_daily_backtest.csv` if available, otherwise the trend pricing file), starting 2002-01-01 with $100,000,000 initial NAV.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

START_DATE = "2002-01-01"
INITIAL_AUM = 100_000_000
W_STOCK = 0.6
W_BOND = 0.4

def resolve_repo_root() -> Path:
    """Notebook may be run from repo root or from Notebooks/."""
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "stock_bond_pricing_data_uncleaned.xlsx").exists():
        return cwd
    if (cwd.parent / "data" / "stock_bond_pricing_data_uncleaned.xlsx").exists():
        return cwd.parent
    raise FileNotFoundError(
        "Could not find data/stock_bond_pricing_data_uncleaned.xlsx "
        "from repo root or parent. Run from project root or Notebooks/."
    )


REPO = resolve_repo_root()
STOCK_BOND_PATH = REPO / "data" / "stock_bond_pricing_data_uncleaned.xlsx"

# Match trend notebook: prefer CSV calendar if you already ran the trend backtest
METHOD1_BACKTEST_CSV_CANDIDATES = [
    REPO / "Notebooks" / "output" / "method1_daily_backtest.csv",
    Path("output") / "method1_daily_backtest.csv",
    Path("Notebooks") / "output" / "method1_daily_backtest.csv",
]

TREND_PRICE_CANDIDATES = [
    REPO / "data" / "trend_model_index_prices_cleaned.xlsx",
    REPO / "trend_model_index_prices_cleaned.xlsx",
]

# Same folder as Method 1 exports (repo-relative, not cwd-sensitive)
OUTPUT_DIR = REPO / "Notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO =", REPO)
print("STOCK_BOND_PATH exists:", STOCK_BOND_PATH.exists())

## Load and clean SPXT / LUATTRUU (Bloomberg-style sheet)

In [ ]:
def load_stock_bond_prices(path: Path) -> pd.DataFrame:
    """
    Expects rows 0–5 metadata / ticker / 'Dates' header, data from row 6.
    Columns after skip: Dates, SPXT PX_LAST, LUATTRUU PX_LAST.
    """
    raw = pd.read_excel(path, header=None, skiprows=6, engine="openpyxl")
    raw.columns = ["Dates", "SPXT Index", "LUATTRUU Index"]
    raw["Dates"] = pd.to_datetime(raw["Dates"], errors="coerce")
    for c in ("SPXT Index", "LUATTRUU Index"):
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
    out = raw.dropna(subset=["Dates"]).copy()
    out = out.set_index("Dates").sort_index()
    out = out[~out.index.duplicated(keep="last")]
    out = out.dropna(how="all").ffill()
    return out


prices_all = load_stock_bond_prices(STOCK_BOND_PATH)
display(prices_all.head())
display(prices_all.tail())
print("Full history:", prices_all.index[0], "→", prices_all.index[-1], "rows:", len(prices_all))

## Align calendar to trend model (Method 1)

In [ ]:
def load_method1_calendar() -> pd.DatetimeIndex | None:
    for p in METHOD1_BACKTEST_CSV_CANDIDATES:
        if p.exists():
            df = pd.read_csv(p, header=[0, 1], index_col=0, parse_dates=True)
            return pd.DatetimeIndex(df.index)
    return None


def load_trend_price_calendar() -> pd.DatetimeIndex:
    data_path = next((p for p in TREND_PRICE_CANDIDATES if p.exists()), None)
    if data_path is None:
        raise FileNotFoundError(
            "Could not find trend_model_index_prices_cleaned.xlsx for calendar fallback."
        )
    df = pd.read_excel(data_path)
    date_col = "Dates" if "Dates" in df.columns else df.columns[0]
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()
    return pd.DatetimeIndex(df.loc[START_DATE:].index)


method1_ix = load_method1_calendar()
if method1_ix is not None:
    trade_calendar = method1_ix
    print("Using Method 1 backtest CSV calendar:", len(trade_calendar), "days")
else:
    trade_calendar = load_trend_price_calendar()
    print("Using trend pricing file calendar (from", START_DATE, "):", len(trade_calendar), "days")

aligned_ix = prices_all.index.intersection(trade_calendar).sort_values()
prices = prices_all.loc[aligned_ix].copy()
prices = prices.loc[START_DATE:]

print("Aligned benchmark:", prices.index[0], "→", prices.index[-1], "rows:", len(prices))

## Daily returns and 60/40 portfolio

In [ ]:
asset_rets = prices.pct_change()
asset_rets.iloc[0] = 0.0

strategy_return = W_STOCK * asset_rets["SPXT Index"] + W_BOND * asset_rets["LUATTRUU Index"]
strategy_return.name = "strategy_return"

nav = INITIAL_AUM * (1.0 + strategy_return).cumprod()
pnl = nav.diff().fillna(0.0)

summary = pd.DataFrame(
    {
        "start_date": [prices.index[0]],
        "end_date": [prices.index[-1]],
        "n_days": [len(prices)],
        "initial_aum": [INITIAL_AUM],
        "end_nav": [nav.iloc[-1]],
        "total_return": [nav.iloc[-1] / nav.iloc[0] - 1.0],
        "w_stock": [W_STOCK],
        "w_bond": [W_BOND],
    }
)
display(summary.T)

bench_flat = pd.DataFrame(
    {
        "spxt_level": prices["SPXT Index"],
        "luattruu_level": prices["LUATTRUU Index"],
        "spxt_return": asset_rets["SPXT Index"],
        "luattruu_return": asset_rets["LUATTRUU Index"],
        "strategy_return": strategy_return,
        "nav": nav,
        "pnl": pnl,
    }
)
display(bench_flat.head())
display(bench_flat.tail())

## Export

Multi-section CSV compatible with `pd.read_csv(..., header=[0, 1])`.

In [ ]:
weights = pd.DataFrame(index=asset_rets.index)
weights["SPXT Index"] = W_STOCK
weights["LUATTRUU Index"] = W_BOND

results_df = pd.DataFrame(
    {
        "strategy_return": strategy_return,
        "nav": nav,
        "pnl": pnl,
        "gross_leverage": 1.0,
        "active_signals": 2.0,
    },
    index=strategy_return.index,
)

daily_output = pd.concat(
    {
        "asset_returns": asset_rets,
        "weights": weights,
        "results": results_df,
    },
    axis=1,
)

out_csv = OUTPUT_DIR / "benchmark_60_40_daily_backtest.csv"
daily_output.to_csv(out_csv)
bench_flat.to_csv(OUTPUT_DIR / "benchmark_60_40_daily_flat.csv")
summary.to_csv(OUTPUT_DIR / "benchmark_60_40_summary.csv", index=False)

print("Wrote:", out_csv.resolve())
print("Wrote:", (OUTPUT_DIR / "benchmark_60_40_daily_flat.csv").resolve())
print("Wrote:", (OUTPUT_DIR / "benchmark_60_40_summary.csv").resolve())

## Sanity checks

In [ ]:
chk = pd.read_csv(out_csv, header=[0, 1], index_col=0, parse_dates=True)
assert chk.index.equals(prices.index), "Index mismatch after round-trip CSV read"
assert chk["results"]["strategy_return"].notna().all()
assert chk["results"]["nav"].notna().all()
assert np.isclose(weights["SPXT Index"], W_STOCK).all()
assert np.isclose(weights["LUATTRUU Index"], W_BOND).all()
print("Sanity checks passed.")